# |멋진 챗봇 만들기 - Transformer 기반 한국어 챗봇

## 0. 라이브러리 임포트

In [1]:
!pip install scikit-learn --break-system-packages

In [66]:
!pip uninstall mecab-python3 python-mecab-ko python-mecab-ko-dic -y
!pip install python-mecab-ko

Found existing installation: mecab-python3 1.0.12
Uninstalling mecab-python3-1.0.12:
  Successfully uninstalled mecab-python3-1.0.12
Found existing installation: python-mecab-ko 1.3.7
Uninstalling python-mecab-ko-1.3.7:
  Successfully uninstalled python-mecab-ko-1.3.7
Found existing installation: python-mecab-ko-dic 2.1.1.post2
Uninstalling python-mecab-ko-dic-2.1.1.post2:
  Successfully uninstalled python-mecab-ko-dic-2.1.1.post2


You can safely remove it manually.


  Using cached python_mecab_ko-1.3.7-cp311-cp311-win_amd64.whl.metadata (3.5 kB)
  Using cached python_mecab_ko_dic-2.1.1.post2-py3-none-any.whl.metadata (1.4 kB)
Using cached python_mecab_ko-1.3.7-cp311-cp311-win_amd64.whl (656 kB)
Using cached python_mecab_ko_dic-2.1.1.post2-py3-none-any.whl (34.5 MB)

   ---------------------------------------- 0/2 [python-mecab-ko-dic]
   ---------------------------------------- 0/2 [python-mecab-ko-dic]
   ---------------------------------------- 2/2 [python-mecab-ko]



In [2]:
!pip list | findstr mecab

python-mecab-ko           1.3.7
python-mecab-ko-dic       2.1.1.post2


In [3]:
from mecab import MeCab
m = MeCab()
print(m.pos("안녕하세요 테스트입니다"))

[('안녕', 'NNG'), ('하', 'XSV'), ('세요', 'EP+EF'), ('테스트', 'NNG'), ('입니다', 'VCP+EC')]


In [4]:
import os
import re
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import sentencepiece as spm

from tqdm.auto import tqdm
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# MeCab (Windows: python-mecab-ko)
from mecab import MeCab
m = MeCab()
# GloVe 증강
import gensim.downloader as api
from gensim.models import KeyedVectors

# BLEU
import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 한글 폰트
import matplotlib.font_manager as fm
for font_path in [
    'C:/Windows/Fonts/malgun.ttf',
    '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
]:
    if os.path.exists(font_path):
        plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
        break

## 하이퍼파라미터 중앙관리

In [5]:
CFG = {
    "DATA_PATH"   : "ChatbotData.csv",
    "PRE_MAX_LEN" : 50,
    "VOCAB_SIZE"  : 3500,
    "D_MODEL"     : 512,        # 참고파일: 512 (기존 368 → 개선)
    "N_LAYERS"    : 2,          # 참고파일: 2   (기존 1   → 개선)
    "N_HEADS"     : 8,
    "D_FF"        : 2048,       # 참고파일: 2048 (기존 1024 → 개선)
    "DROPOUT"     : 0.3,
    "WARMUP_STEPS": 4000,       # 논문 기본값 (기존 1000 → 개선)
    "BATCH_SIZE"  : 64,
    "EPOCHS"      : 20,
    "DEVICE"      : torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

print(f"사용 디바이스: {CFG['DEVICE']}")
print(f"PyTorch 버전 : {torch.__version__}")

mecab = MeCab()
print(f"MeCab 테스트: {[t.surface for t in mecab.parse('안녕하세요') if t.surface.strip()]}")

사용 디바이스: cpu
PyTorch 버전 : 2.10.0+cpu
MeCab 테스트: ['안녕', '하', '세요']


## Step 1. 데이터 로드

In [6]:
def load_chatbot_data(data_path=CFG["DATA_PATH"]):
    if not os.path.exists(data_path):
        import urllib.request
        print("ChatbotData.csv 다운로드 중...")
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv",
            data_path
        )
    df = pd.read_csv(data_path)
    print(f"[Step 1] 전체 데이터 수: {len(df)}")
    print(df.head(3))
    return df

## Step 2. 데이터 정제

In [7]:
def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'\.{2,}', ' . ', sentence)
    sentence = re.sub(r' {2,}', ' ', sentence)
    sentence = re.sub(r"[^a-zA-Z0-9ㄱ-ㅎㅏ-ㅣ가-힣?.!,]+", " ", sentence)
    return sentence.strip()


## Step 3. 토큰화 + 코퍼스 구축

In [8]:
def tokenize_mecab(sentence):
    return [t.surface for t in mecab.parse(sentence) if t.surface.strip()]


def build_corpus(src_sentences, tgt_sentences, max_len=50):
    que_corpus, ans_corpus = [], []
    seen = set()

    for src, tgt in zip(src_sentences, tgt_sentences):
        src_tokens = tokenize_mecab(preprocess_sentence(src))
        tgt_tokens = tokenize_mecab(preprocess_sentence(tgt))

        if len(src_tokens) <= max_len and len(tgt_tokens) <= max_len:
            pair = (tuple(src_tokens), tuple(tgt_tokens))
            if pair not in seen:
                seen.add(pair)
                que_corpus.append(src_tokens)
                ans_corpus.append(tgt_tokens)

    return que_corpus, ans_corpus


## Step 4. 데이터 증강

In [9]:
def load_word_vectors():
    kv_path = "data/ko.kv"
    if os.path.exists(kv_path):
        wv = KeyedVectors.load(kv_path)
        print("[Step 4] ko.kv 로드 완료")
    else:
        print("[Step 4] glove-wiki-gigaword-300 로드 중...")
        wv = api.load('glove-wiki-gigaword-300')
    return wv


def lexical_sub(tokens, wv):
    try:
        pool   = [t for t in tokens if t in wv]
        if not pool:
            return tokens
        target = random.choice(pool)
        sub    = random.choice(wv.most_similar(target, topn=10))[0]
        return [sub if t == target else t for t in tokens]
    except Exception:
        return tokens


def augment_data(que_corpus, ans_corpus, wv):
    added_que, added_ans = [], []
    for i in tqdm(range(len(que_corpus)), desc="증강 중"):
        added_que.append(lexical_sub(que_corpus[i], wv))
        added_ans.append(ans_corpus[i])
        added_que.append(que_corpus[i])
        added_ans.append(lexical_sub(ans_corpus[i], wv))

    que_corpus += added_que
    ans_corpus += added_ans
    print(f"[Step 4] 증강 완료: {len(que_corpus)}개 (약 3배)")
    return que_corpus, ans_corpus


## Step 5. 벡터화

In [10]:
def generate_tokenizer(corpus, vocab_size, lang):
    temp_file = f'chatbot_{lang}.txt'
    with open(temp_file, 'w', encoding='utf-8') as f:
        for tokens in corpus:
            f.write(" ".join(tokens) + '\n')

    spm.SentencePieceTrainer.train(
        input=temp_file, model_prefix=lang, vocab_size=vocab_size,
        model_type='bpe', pad_id=0, bos_id=1, eos_id=2, unk_id=3
    )
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.load(f'{lang}.model')
    return tokenizer


def make_corpus(sentences, tokenizer):
    """형태소 리스트 → " ".join → SentencePiece 인코딩 (참고파일 동일)"""
    return [tokenizer.encode_as_ids(" ".join(s)) for s in sentences]


def pad_sequences_torch(sequences, maxlen, padding_value=0):
    """tensorflow pad_sequences 대체 (torch 순수 구현)"""
    result = torch.zeros(len(sequences), maxlen, dtype=torch.long)
    for i, seq in enumerate(sequences):
        length = min(len(seq), maxlen)
        result[i, :length] = torch.tensor(seq[:length], dtype=torch.long)
    return result


## Step 6-A. Transformer 모듈

In [11]:
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    sinusoid_table = np.array([[cal_angle(p, i) for i in range(d_model)] for p in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.depth     = d_model // num_heads
        self.d_model   = d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scaled_qk = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(Q.size(-1))
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)   # [Loss 개선] mask * -1e9
        return torch.matmul(F.softmax(scaled_qk, dim=-1), V)

    def split_heads(self, x):
        b, l, _ = x.size()
        return x.view(b, l, self.num_heads, self.depth).permute(0, 2, 1, 3)

    def combine_heads(self, x):
        b, _, l, _ = x.size()
        return x.permute(0, 2, 1, 3).contiguous().view(b, l, self.d_model)

    def forward(self, Q, K, V, mask=None):
        out = self.scaled_dot_product_attention(
            self.split_heads(self.W_q(Q)),
            self.split_heads(self.W_k(K)),
            self.split_heads(self.W_v(V)),
            mask
        )
        return self.linear(self.combine_heads(out))


class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, n_heads)
        self.ffn     = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1   = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2   = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.dropout(self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)) + x
        x = self.dropout(self.ffn(self.norm2(x))) + x
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.enc_attn  = MultiHeadAttention(d_model, n_heads)
        self.ffn       = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1     = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2     = nn.LayerNorm(d_model, eps=1e-6)
        self.norm3     = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, dec_mask):
        x = self.dropout(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), dec_mask)) + x
        x = self.dropout(self.enc_attn(self.norm2(x), enc_out, enc_out, dec_enc_mask)) + x
        x = self.dropout(self.ffn(self.norm3(x))) + x
        return x


class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len, dropout=0.3):
        super().__init__()
        self.d_model = float(d_model)
        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.register_buffer("pos_enc",
            torch.tensor(positional_encoding(pos_len, d_model), dtype=torch.float32))
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc      = nn.Linear(d_model, tgt_vocab_size)

    def embed(self, emb, x):
        return self.dropout(emb(x) * math.sqrt(self.d_model) + self.pos_enc[:x.size(1)].unsqueeze(0))

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        x = self.embed(self.enc_emb, enc_in)
        for layer in self.encoder:
            x = layer(x, enc_mask)
        enc_out = x

        y = self.embed(self.dec_emb, dec_in)
        for layer in self.decoder:
            y = layer(y, enc_out, dec_enc_mask, dec_mask)
        return self.fc(y)

## Step 6-C. 마스킹

In [12]:
def generate_padding_mask(seq):
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size, device):
    return torch.triu(torch.ones(size, size, device=device), diagonal=1)


def generate_masks(src, tgt):
    device       = src.device
    enc_mask     = generate_padding_mask(src).to(device)
    dec_enc_mask = generate_padding_mask(src).to(device)
    dec_mask     = torch.max(
        generate_padding_mask(tgt).to(device),
        generate_lookahead_mask(tgt.shape[1], device).unsqueeze(0).unsqueeze(1)
    )
    return enc_mask, dec_enc_mask, dec_mask

## Step 6-D. LR Scheduler

In [13]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model      = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = max(1.0, float(step))
        return (self.d_model ** -0.5) * min(step ** -0.5, step * self.warmup_steps ** -1.5)

## Step 6-E. Loss 함수

In [14]:
def loss_function(real, pred):
    mask  = (real != 0).float()
    loss_ = F.cross_entropy(pred.reshape(-1, pred.size(-1)), real.reshape(-1), reduction='none')
    return (loss_ * mask.reshape(-1)).sum() / mask.sum()

## Step 6-F. 학습 루프

In [15]:
def train_model(model, train_loader, val_loader, device):
    lr_sch    = LearningRateScheduler(CFG["D_MODEL"], CFG["WARMUP_STEPS"])
    optimizer = optim.Adam(model.parameters(), lr=lr_sch(1), betas=(0.9, 0.98), eps=1e-9)

    for epoch in range(CFG["EPOCHS"]):
        model.train()
        total_train_loss = 0.0
        bar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{CFG['EPOCHS']} [Train]")

        for batch_idx, (src, tgt) in enumerate(bar):
            src, tgt = src.to(device), tgt.to(device)
            tgt_in, gold = tgt[:, :-1], tgt[:, 1:]
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

            # [Loss 개선] global_step 정확히 계산
            global_step = epoch * len(train_loader) + batch_idx + 1
            for pg in optimizer.param_groups:
                pg['lr'] = lr_sch(global_step)

            optimizer.zero_grad()
            pred = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
            loss = loss_function(gold, pred)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            bar.set_postfix(loss=f"{loss.item():.4f}")

        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                tgt_in, gold = tgt[:, :-1], tgt[:, 1:]
                enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
                pred = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
                total_val_loss += loss_function(gold, pred).item()

        print(f"▶ Epoch {epoch+1:2d} | "
              f"Train Loss: {total_train_loss/len(train_loader):.4f} | "
              f"Val Loss: {total_val_loss/len(val_loader):.4f}")

    print("\n[Step 6] 학습 완료!")

## Step 7. 추론 (Greedy + Beam Search)

In [16]:
def _encode_input(sentence, tokenizer):
    tokens = tokenize_mecab(preprocess_sentence(sentence))
    ids    = tokenizer.encode_as_ids(" ".join(tokens))
    return torch.LongTensor([ids])


def chatbot_inference(sentence, tokenizer, model, device, max_len=50):
    model.eval()
    src = _encode_input(sentence, tokenizer).to(device)
    tgt = [1]

    for _ in range(max_len):
        tgt_t = torch.LongTensor([tgt]).to(device)
        enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_t)
        with torch.no_grad():
            pred = model(src, tgt_t, enc_mask, dec_enc_mask, dec_mask)
        next_id = torch.argmax(pred[:, -1:, :], dim=-1).item()
        tgt.append(next_id)
        if next_id == 2:
            break

    return tokenizer.decode_ids([i for i in tgt if i > 3])


def beam_search_decoder(sentence, tokenizer, model, device, beam_size=5, max_len=50):
    model.eval()
    src   = _encode_input(sentence, tokenizer).to(device)
    beams = [([1], 0.0)]

    for _ in range(max_len):
        new_beams = []
        for ids, score in beams:
            if ids[-1] == 2:
                new_beams.append((ids, score))
                continue
            tgt_t = torch.LongTensor([ids]).to(device)
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_t)
            with torch.no_grad():
                pred = model(src, tgt_t, enc_mask, dec_enc_mask, dec_mask)
            log_probs    = F.log_softmax(pred[:, -1:, :], dim=-1).squeeze()
            top_v, top_i = torch.topk(log_probs, beam_size)
            for i in range(beam_size):
                new_beams.append((ids + [top_i[i].item()], score + top_v[i].item()))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
        if all(ids[-1] == 2 for ids, _ in beams):
            break

    return tokenizer.decode_ids([i for i in beams[0][0] if i > 3])


def chat(sentence, tokenizer, model, device, use_beam=True):
    result = beam_search_decoder(sentence, tokenizer, model, device) \
             if use_beam else chatbot_inference(sentence, tokenizer, model, device)
    print(f"Q: {sentence}\nA: {result}\n")
    return result


def calculate_bleu(enc_test, dec_test, tokenizer, model, device):
    smoothie = SmoothingFunction().method1
    scores   = []
    for i in tqdm(range(len(enc_test)), desc="BLEU 계산"):
        ref       = tokenizer.decode_ids([x for x in dec_test[i] if x > 3]).split()
        hyp       = beam_search_decoder(
            tokenizer.decode_ids(enc_test[i]), tokenizer, model, device
        ).split()
        scores.append(sentence_bleu([ref], hyp, smoothing_function=smoothie))
    avg = np.mean(scores)
    print(f"[Step 7] 평균 BLEU Score: {avg * 100:.2f}점")
    return avg

## 메인 실행

In [17]:
if __name__ == "__main__":

    print("=" * 60)
    print("  Transformer 한국어 챗봇 학습")
    print("=" * 60)

    # Step 1
    df = load_chatbot_data()

    # Step 3 — 분할 먼저 (참고파일 방식)
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
    val_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42)
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # Step 3 — train만 토크나이저 학습용 코퍼스 구축
    que_corpus, ans_corpus = build_corpus(
        train_df['Q'].tolist(), train_df['A'].tolist(), CFG["PRE_MAX_LEN"]
    )

    # Step 4 — 증강 (선택, ko.kv 없으면 주석 해제 후 진행)
    # wv = load_word_vectors()
    # que_corpus, ans_corpus = augment_data(que_corpus, ans_corpus, wv)

    # Step 5 — SentencePiece 토크나이저 (train만으로 학습)
    tokenizer = generate_tokenizer(que_corpus + ans_corpus, CFG["VOCAB_SIZE"], 'kor')
    tokenizer.set_encode_extra_options("bos:eos")

    enc_train = make_corpus(que_corpus, tokenizer)
    dec_train = make_corpus(ans_corpus, tokenizer)

    val_que, val_ans   = build_corpus(val_df['Q'].tolist(),  val_df['A'].tolist())
    enc_val            = make_corpus(val_que, tokenizer)
    dec_val            = make_corpus(val_ans, tokenizer)

    test_que, test_ans = build_corpus(test_df['Q'].tolist(), test_df['A'].tolist())
    enc_test           = make_corpus(test_que, tokenizer)
    dec_test           = make_corpus(test_ans, tokenizer)

    MAX_LEN = CFG["PRE_MAX_LEN"]
    device  = CFG["DEVICE"]

    train_inputs  = pad_sequences_torch(enc_train, MAX_LEN).to(device)
    train_outputs = pad_sequences_torch(dec_train, MAX_LEN).to(device)
    val_inputs    = pad_sequences_torch(enc_val,   MAX_LEN).to(device)
    val_outputs   = pad_sequences_torch(dec_val,   MAX_LEN).to(device)
    test_inputs   = pad_sequences_torch(enc_test,  MAX_LEN).to(device)
    test_outputs  = pad_sequences_torch(dec_test,  MAX_LEN).to(device)

    print(f"train: {train_inputs.shape} | val: {val_inputs.shape} | test: {test_inputs.shape}")

    train_loader = DataLoader(TensorDataset(train_inputs, train_outputs),
                              batch_size=CFG["BATCH_SIZE"], shuffle=True)
    val_loader   = DataLoader(TensorDataset(val_inputs, val_outputs),
                              batch_size=CFG["BATCH_SIZE"], shuffle=False)

    # Step 6 — 모델
    model = Transformer(
        n_layers      = CFG["N_LAYERS"],
        d_model       = CFG["D_MODEL"],
        n_heads       = CFG["N_HEADS"],
        d_ff          = CFG["D_FF"],
        src_vocab_size= CFG["VOCAB_SIZE"],
        tgt_vocab_size= CFG["VOCAB_SIZE"],
        pos_len       = CFG["PRE_MAX_LEN"],
        dropout       = CFG["DROPOUT"],
    ).to(device)

    print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    train_model(model, train_loader, val_loader, device)

    # Step 7 — 테스트
    print("\n" + "=" * 60)
    for q in ["반가워요", "오늘 날씨 어때?", "배고픈데 뭐 먹을까?", "졸려요", "고마워요"]:
        chat(q, tokenizer, model, device)

    calculate_bleu(enc_test, dec_test, tokenizer, model, device)

    torch.save({'model_state_dict': model.state_dict(), 'cfg': CFG}, "chatbot_transformer.pt")
    print("모델 저장 완료!")

  Transformer 한국어 챗봇 학습
[Step 1] 전체 데이터 수: 11823
              Q            A  label
0        12시 땡!   하루가 또 가네요.      0
1   1지망 학교 떨어졌어    위로해 드립니다.      0
2  3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
Train: 9458, Val: 1182, Test: 1183
train: torch.Size([9412, 50]) | val: torch.Size([1182, 50]) | test: torch.Size([1182, 50])
파라미터 수: 20,092,332


Epoch  1/20 [Train]:   0%|          | 0/148 [00:00<?, ?it/s]

▶ Epoch  1 | Train Loss: 55.8816 | Val Loss: 34.7825


Epoch  2/20 [Train]:   0%|          | 0/148 [00:00<?, ?it/s]

▶ Epoch  2 | Train Loss: 39.7356 | Val Loss: 25.1636


...

▶ Epoch 19 | Train Loss: 5.4296 | Val Loss: 6.0719


Epoch 20/20 [Train]:   0%|          | 0/148 [00:00<?, ?it/s]

▶ Epoch 20 | Train Loss: 5.2378 | Val Loss: 5.9053

[Step 6] 학습 완료!

Q: 반가워요
A: 서로 공감 할 때 조심 하 세요 .

Q: 오늘 날씨 어때?
A: 동심 으로 돌아갈 거 예요 .

Q: 배고픈데 뭐 먹을까?
A: 왕이너 에게 해 보 세요 .

Q: 졸려요
A: 스트레스 받 아 보 세요 .

Q: 고마워요
A: 스트레스 받 아 보 세요 .



BLEU 계산:   0%|          | 0/1182 [00:00<?, ?it/s]

[Step 7] 평균 BLEU Score: 3.98점
모델 저장 완료!
